# Notebook 03 — Model Architecture

## Building a Foundation Model from Scratch

This notebook defines, implements, and validates the decoder-only Transformer architecture used in the controlled model-scaling experiment.

### Experimental architecture family

We will implement three progressively larger members of the same architectural family while holding the tokenizer, dataset, context length, training objective, and training methodology constant.

| Model | Layers | d_model | Heads | Head Dim | SwiGLU Hidden | Target Scale |
|---|---:|---:|---:|---:|---:|---:|
| A | 4 | 256 | 4 | 64 | 704 | ~7M |
| B | 6 | 384 | 6 | 64 | 1,024 | ~17M |
| C | 8 | 512 | 8 | 64 | 1,360 | ~34M |

### Core architecture

Each model uses:

- learned token embeddings
- causal multi-head self-attention
- Rotary Position Embeddings (RoPE)
- RMSNorm
- SwiGLU feed-forward networks
- pre-normalization residual blocks
- final RMSNorm
- tied token-embedding / output-projection weights

The implementation is written explicitly in PyTorch rather than using a prebuilt Transformer model.

### Fixed model-level controls

- Vocabulary size: 16,384
- Context length: 512 tokens
- Dropout: 0.10
- Attention head dimension: 64
- Linear projection biases: disabled
- Input/output embedding weights: tied


In [1]:
from dataclasses import dataclass

VOCAB_SIZE = 16_384
CONTEXT_LENGTH = 512
DROPOUT = 0.10


@dataclass(frozen=True)
class ModelConfig:
    name: str
    vocab_size: int
    context_length: int
    n_layers: int
    d_model: int
    n_heads: int
    d_ff: int
    dropout: float = DROPOUT

    @property
    def head_dim(self) -> int:
        return self.d_model // self.n_heads


MODEL_CONFIGS = {
    "A": ModelConfig(
        name="Model A",
        vocab_size=VOCAB_SIZE,
        context_length=CONTEXT_LENGTH,
        n_layers=4,
        d_model=256,
        n_heads=4,
        d_ff=704,
    ),
    "B": ModelConfig(
        name="Model B",
        vocab_size=VOCAB_SIZE,
        context_length=CONTEXT_LENGTH,
        n_layers=6,
        d_model=384,
        n_heads=6,
        d_ff=1024,
    ),
    "C": ModelConfig(
        name="Model C",
        vocab_size=VOCAB_SIZE,
        context_length=CONTEXT_LENGTH,
        n_layers=8,
        d_model=512,
        n_heads=8,
        d_ff=1360,
    ),
}

MODEL_CONFIGS


{'A': ModelConfig(name='Model A', vocab_size=16384, context_length=512, n_layers=4, d_model=256, n_heads=4, d_ff=704, dropout=0.1),
 'B': ModelConfig(name='Model B', vocab_size=16384, context_length=512, n_layers=6, d_model=384, n_heads=6, d_ff=1024, dropout=0.1),
 'C': ModelConfig(name='Model C', vocab_size=16384, context_length=512, n_layers=8, d_model=512, n_heads=8, d_ff=1360, dropout=0.1)}

In [2]:
for key, cfg in MODEL_CONFIGS.items():
    assert cfg.d_model % cfg.n_heads == 0
    assert cfg.head_dim == 64
    assert cfg.context_length == CONTEXT_LENGTH
    assert cfg.vocab_size == VOCAB_SIZE

    print(
        f"{cfg.name}: "
        f"{cfg.n_layers} layers, "
        f"d_model={cfg.d_model}, "
        f"{cfg.n_heads} heads × {cfg.head_dim} dims, "
        f"d_ff={cfg.d_ff}"
    )


Model A: 4 layers, d_model=256, 4 heads × 64 dims, d_ff=704
Model B: 6 layers, d_model=384, 6 heads × 64 dims, d_ff=1024
Model C: 8 layers, d_model=512, 8 heads × 64 dims, d_ff=1360


In [3]:
def analytical_parameter_count(cfg: ModelConfig) -> dict:
    embeddings = cfg.vocab_size * cfg.d_model
    attention_per_layer = 4 * cfg.d_model**2
    swiglu_per_layer = 3 * cfg.d_model * cfg.d_ff
    norms_per_layer = 2 * cfg.d_model
    block_per_layer = attention_per_layer + swiglu_per_layer + norms_per_layer
    transformer_blocks = cfg.n_layers * block_per_layer
    final_norm = cfg.d_model
    total = embeddings + transformer_blocks + final_norm
    return {"embeddings": embeddings, "attention_per_layer": attention_per_layer, "swiglu_per_layer": swiglu_per_layer, "norms_per_layer": norms_per_layer, "block_per_layer": block_per_layer, "transformer_blocks": transformer_blocks, "final_norm": final_norm, "total": total}

for key, cfg in MODEL_CONFIGS.items():
    counts = analytical_parameter_count(cfg)
    print(f"{cfg.name}: {counts['total']:,} parameters ({counts['total'] / 1e6:.2f}M)")


Model A: 7,407,872 parameters (7.41M)
Model B: 16,913,280 parameters (16.91M)
Model C: 33,497,600 parameters (33.50M)


## Parameter-count mental model

For each Transformer block:

- Attention contributes `4 * d_model^2` parameters for Q, K, V, and output projections.
- SwiGLU contributes `3 * d_model * d_ff` parameters for gate, up, and down projections.
- Two RMSNorms contribute `2 * d_model` learned scale parameters.

The token embedding matrix contributes `vocab_size * d_model` parameters and is tied to the language-model output projection.

RoPE adds no learned parameters.

For Model A, the embedding matrix alone contains 4,194,304 parameters, which is about 57% of the full 7.41M-parameter model. This is why vocabulary size and weight tying matter materially at small model scales.


## Chunk 2 — RMSNorm

Before implementing attention or the feed-forward network, we implement the normalization used throughout the model.

### Why normalization is needed

As activations move through many residual blocks, their scale can drift. Normalization keeps the numerical scale of those activations controlled, which generally makes optimization more stable.

A classic **LayerNorm** normalizes using both the mean and variance of the hidden features:

$$\mathrm{LayerNorm}(x)=\gamma\odot\frac{x-\mu}{\sqrt{\sigma^2+\epsilon}}+\beta$$

**RMSNorm** is simpler. It does not subtract the mean and does not use an additive bias term. It rescales the vector using its root-mean-square magnitude:

$$\mathrm{RMS}(x)=\sqrt{\frac{1}{d}\sum_{i=1}^{d}x_i^2}$$

$$\mathrm{RMSNorm}(x)=g\odot\frac{x}{\sqrt{\frac{1}{d}\sum_{i=1}^{d}x_i^2+\epsilon}}$$

where `g` is a learned scale vector with one parameter for each hidden dimension.

### Parameter consequence

For a model width `d_model`, one RMSNorm contributes exactly `d_model` learned parameters.

Our architecture uses two RMSNorms inside every Transformer block and one final RMSNorm after the last block.

### Pre-normalization placement

```text
x = x + Attention(RMSNorm(x))
x = x + SwiGLU(RMSNorm(x))
```


In [ ]:
import torch
import torch.nn as nn

class RMSNorm(nn.Module):
    def __init__(self, d_model: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d_model))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        input_dtype = x.dtype
        x_float = x.float()
        mean_square = x_float.pow(2).mean(dim=-1, keepdim=True)
        x_normalized = x_float * torch.rsqrt(mean_square + self.eps)
        return (x_normalized * self.weight).to(dtype=input_dtype)


In [ ]:
torch.manual_seed(42)
cfg = MODEL_CONFIGS["A"]
norm = RMSNorm(cfg.d_model)
x = torch.randn(2, 5, cfg.d_model)
y = norm(x)
assert y.shape == x.shape
parameter_count = sum(p.numel() for p in norm.parameters())
assert parameter_count == cfg.d_model
output_rms = y.float().pow(2).mean(dim=-1).sqrt()
max_deviation_from_one = (output_rms - 1.0).abs().max().item()
assert max_deviation_from_one < 1e-5
print(f"Input shape:              {tuple(x.shape)}")
print(f"Output shape:             {tuple(y.shape)}")
print(f"RMSNorm parameters:       {parameter_count:,}")
print(f"Expected parameters:      {cfg.d_model:,}")
print(f"Max RMS deviation from 1: {max_deviation_from_one:.2e}")


In [ ]:
for key, cfg in MODEL_CONFIGS.items():
    test_norm = RMSNorm(cfg.d_model)
    observed = sum(p.numel() for p in test_norm.parameters())
    expected = cfg.d_model
    assert observed == expected
    print(f"{cfg.name}: RMSNorm = {observed:,} learned parameters")


### RMSNorm mental model

> **RMSNorm controls the magnitude of the hidden-state vector without recentering it.**

It asks: *How large is this vector on average?* Then it rescales the vector to a controlled magnitude and lets the learned scale `g` determine the useful magnitude of each feature dimension.

RMSNorm does **not** mix information across tokens. Each token's hidden vector is normalized independently across its feature dimension.
